In [44]:
import pandas as pd

RAW_PATH = "../data_Lake/raw/Sample_Data_Ingestion.csv"

df = pd.read_csv(RAW_PATH)

print("Initial Shape:", df.shape)
df.head()

Initial Shape: (200, 16)


,Server_ID,Hostname,IP_Address,OS_Type,Server_Location,CPU_Utilization (%),Memory_Usage (%),Disk_IO (%),Network_Traffic_In (MB/s),Network_Traffic_Out (MB/s),Uptime (Hours),Downtime (Hours),Admin_Name,Admin_Email,Admin_Phone,Log_Timestamp
0,536c9e75-40a1-4a74-8ca6-c6abaec13999,srv-web02,100.33.3.46,Windows Server 2022,"Berlin, Germany",28.10,88.60,28.36,31.13,52.54,18.49,5.51,John Duarte,laurasolomon@example.net,+77 1834560143,11-03-2025 16:45
1,26bae71d-c025-4082-9484-ef7aa9e04836,srv-web02,249.168.171.15,Ubuntu 20.04,"London, UK",49.72,84.35,73.89,123.34,97.10,19.71,4.29,Abigail Montgomery,sherripatterson@example.com,+10 3791349757,25-01-2025 10:59
2,454de38a-5961-490f-9d46-d20663a36baf,srv-backup02,248.247.52.48,Red Hat Enterprise Linux,"New York, USA",44.72,57.64,14.94,69.60,112.81,18.64,5.36,David Jones,veronicanavarro@example.net,+38 9503876034,29-01-2025 08:31
3,8b8a3f40-1b67-4fe6-a65e-93fd07f317f7,srv-web02,87.221.187.138,Red Hat Enterprise Linux,"New York, USA",37.69,97.97,71.46,62.97,148.51,18.14,5.86,Dominic Richards,frankdawson@example.org,+93 2330701744,01-03-2025 16:49
4,375cb3db-89d3-4705-af4a-8f829cce1928,srv-app02,168.8.91.217,Windows Server 2022,"Tokyo, Japan",63.97,67.31,96.46,45.33,131.50,22.78,1.22,Paul Edwards,hgibson@example.com,+29 9075330226,21-02-2025 11:02


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Server_ID                   200 non-null    object 
 1   Hostname                    200 non-null    object 
 2   IP_Address                  200 non-null    object 
 3   OS_Type                     200 non-null    object 
 4   Server_Location             200 non-null    object 
 5   CPU_Utilization (%)         200 non-null    float64
 6   Memory_Usage (%)            200 non-null    float64
 7   Disk_IO (%)                 200 non-null    float64
 8   Network_Traffic_In (MB/s)   200 non-null    float64
 9   Network_Traffic_Out (MB/s)  200 non-null    float64
 10  Uptime (Hours)              200 non-null    float64
 11  Downtime (Hours)            200 non-null    float64
 12  Admin_Name                  200 non-null    object 
 13  Admin_Email                 200 non

In [46]:
df.duplicated().sum()

0

In [47]:
df["Log_Timestamp"] = pd.to_datetime(df["Log_Timestamp"], errors="coerce")

print("Invalid timestamps:", df["Log_Timestamp"].isna().sum())
df["Log_Timestamp"].head()

Invalid timestamps: 124


0   2025-11-03 16:45:00
1                   NaT
2                   NaT
3   2025-01-03 16:49:00
4                   NaT
Name: Log_Timestamp, dtype: datetime64[ns]

In [48]:
# correcting the timestamp 
df["Log_Timestamp"] = pd.to_datetime(
    df["Log_Timestamp"],
    dayfirst=True,
    errors="coerce"
)

print("Invalid timestamps after fix:", df["Log_Timestamp"].isna().sum())

Invalid timestamps after fix: 124


In [49]:
df["Log_Timestamp"].astype(str).unique()[:20]

array(['2025-11-03 16:45:00', 'NaT', '2025-01-03 16:49:00',
       '2025-09-03 14:58:00', '2025-06-03 10:06:00',
       '2025-07-02 18:43:00', '2025-10-02 15:01:00',
       '2025-06-02 00:58:00', '2025-07-02 05:21:00',
       '2025-08-03 10:54:00', '2025-06-02 21:54:00',
       '2025-08-03 05:21:00', '2025-02-02 06:32:00',
       '2025-09-03 22:41:00', '2025-01-03 16:46:00',
       '2025-08-02 06:50:00', '2025-04-02 09:33:00',
       '2025-01-03 07:24:00', '2025-02-02 00:40:00',
       '2025-02-02 17:31:00'], dtype=object)

In [7]:
df = pd.read_csv("../data_Lake/raw/Sample_Data_Ingestion.csv")

# Replace string 'NaT' with real NaN
df["Log_Timestamp"] = df["Log_Timestamp"].replace("NaT", pd.NA)

# Now parse timestamp
df["Log_Timestamp"] = pd.to_datetime(df["Log_Timestamp"], errors="coerce")

print("Invalid timestamps:", df["Log_Timestamp"].isna().sum())

Invalid timestamps: 124


In [50]:
# creating timestamp flag 

df["timestamp_missing_flag"] = df["Log_Timestamp"].isna().astype(int)

In [51]:
# creating time features 
df["log_date"] = df["Log_Timestamp"].dt.date
df["log_hour"] = df["Log_Timestamp"].dt.hour
df["log_month"] = df["Log_Timestamp"].dt.month
df["log_weekday"] = df["Log_Timestamp"].dt.day_name()

In [52]:
# Server Load Category
def cpu_load_category(cpu):
    if cpu < 40:
        return "Low"
    elif cpu < 70:
        return "Medium"
    else:
        return "High"

df["cpu_load_category"] = df["CPU_Utilization (%)"].apply(cpu_load_category)

In [53]:
# Memory Pressure Indicator 
def memory_pressure(mem):
    if mem < 60:
        return "Normal"
    elif mem < 80:
        return "Warning"
    else:
        return "Critical"

df["memory_pressure"] = df["Memory_Usage (%)"].apply(memory_pressure)

In [54]:
# Network Stress Score
df["network_stress_score"] = (
    df["Network_Traffic_In (MB/s)"] +
    df["Network_Traffic_Out (MB/s)"]
) / 2

In [55]:
# Availability Percentage
df["availability_pct"] = (
    df["Uptime (Hours)"] /
    (df["Uptime (Hours)"] + df["Downtime (Hours)"])
) * 100

In [56]:
# Downtime flag
df["downtime_flag"] = (df["Downtime (Hours)"] > 0).astype(int)

In [57]:
df["performance_score"] = (
    (100 - df["CPU_Utilization (%)"]) * 0.4 +
    (100 - df["Memory_Usage (%)"]) * 0.3 +
    (100 - df["Disk_IO (%)"]) * 0.3
)

In [58]:
df.head()

,Server_ID,Hostname,IP_Address,OS_Type,Server_Location,CPU_Utilization (%),Memory_Usage (%),Disk_IO (%),Network_Traffic_In (MB/s),Network_Traffic_Out (MB/s),...,log_date,log_hour,log_month,log_weekday,cpu_load_category,memory_pressure,network_stress_score,availability_pct,downtime_flag,performance_score
0,536c9e75-40a1-4a74-8ca6-c6abaec13999,srv-web02,100.33.3.46,Windows Server 2022,"Berlin, Germany",28.10,88.60,28.36,31.13,52.54,...,2025-11-03,16.0,11.0,Monday,Low,Critical,41.835,77.041667,1,53.672
1,26bae71d-c025-4082-9484-ef7aa9e04836,srv-web02,249.168.171.15,Ubuntu 20.04,"London, UK",49.72,84.35,73.89,123.34,97.10,...,NaT,NaN,NaN,NaN,Medium,Critical,110.220,82.125000,1,32.640
2,454de38a-5961-490f-9d46-d20663a36baf,srv-backup02,248.247.52.48,Red Hat Enterprise Linux,"New York, USA",44.72,57.64,14.94,69.60,112.81,...,NaT,NaN,NaN,NaN,Medium,Normal,91.205,77.666667,1,60.338
3,8b8a3f40-1b67-4fe6-a65e-93fd07f317f7,srv-web02,87.221.187.138,Red Hat Enterprise Linux,"New York, USA",37.69,97.97,71.46,62.97,148.51,...,2025-01-03,16.0,1.0,Friday,Low,Critical,105.740,75.583333,1,34.095
4,375cb3db-89d3-4705-af4a-8f829cce1928,srv-app02,168.8.91.217,Windows Server 2022,"Tokyo, Japan",63.97,67.31,96.46,45.33,131.50,...,NaT,NaN,NaN,NaN,Medium,Warning,88.415,94.916667,1,25.281


In [60]:
# the server summary table

server_summary = df.groupby("Server_ID").agg({
    "CPU_Utilization (%)": "mean",
    "Memory_Usage (%)": "mean",
    "Disk_IO (%)": "mean",
    "network_stress_score": "mean",
    "availability_pct": "mean",
    "performance_score": "mean",
    "downtime_flag": "sum"
}).reset_index()

server_summary.rename(columns={
    "downtime_flag": "downtime_events"
}, inplace=True)

server_summary.head()

,Server_ID,CPU_Utilization (%),Memory_Usage (%),Disk_IO (%),network_stress_score,availability_pct,performance_score,downtime_events
0,0435b059-4f09-4a27-abfc-2a2bd4548aa7,87.98,34.46,66.78,93.740,87.708333,34.436,1
1,04799af0-d18c-4d82-925b-cdcc9b69ee0c,51.42,54.94,32.41,42.020,79.750000,53.227,1
2,0564a58b-bc34-4c9c-9cdc-907fc1708c67,32.97,86.84,22.00,94.005,87.541667,54.160,1
3,05dcde9e-836e-4169-b3dc-519478cbafa0,46.02,31.73,48.49,71.930,81.416667,57.526,1
4,07b5619c-8ed1-4226-a35c-880b76379cab,63.36,94.26,27.60,100.630,91.166667,38.098,1


In [61]:
# Location level Aggregation 

location_summary = df.groupby("Server_Location").agg({
    "CPU_Utilization (%)": "mean",
    "Memory_Usage (%)": "mean",
    "availability_pct": "mean",
    "performance_score": "mean"
}).reset_index()

location_summary.head()

,Server_Location,CPU_Utilization (%),Memory_Usage (%),availability_pct,performance_score
0,"Berlin, Germany",54.360256,60.572308,86.126068,43.322051
1,"London, UK",62.509535,65.326512,86.567829,40.055047
2,"New York, USA",56.966250,66.279250,84.311458,40.077475
3,"Sydney, Australia",51.552222,50.672500,86.238426,48.402278
4,"Tokyo, Japan",59.959048,56.708810,86.152778,41.401952


In [63]:
# Time trend Aggregation 
time_summary = df.groupby("log_date").agg({
    "CPU_Utilization (%)": "mean",
    "Memory_Usage (%)": "mean",
    "availability_pct": "mean",
    "performance_score": "mean"
}).reset_index()

time_summary.head()

,log_date,CPU_Utilization (%),Memory_Usage (%),availability_pct,performance_score
0,2025-01-02,68.886667,44.193333,82.055556,41.632333
1,2025-01-03,67.242500,83.540000,81.395833,30.976250
2,2025-02-02,72.657143,46.020000,83.529762,37.312714
3,2025-02-03,54.530000,58.782000,82.991667,47.368400
4,2025-03-02,59.185000,37.175000,85.604167,56.181000


In [65]:
server_summary.to_csv("../data_Lake/processed/server_summary.csv", index=False)

location_summary.to_csv("../data_Lake/processed/location_summary.csv", index=False)

time_summary.to_csv("../data_Lake/processed/time_summary.csv", index=False)

df.to_csv("../data_Lake/processed/server_metrics_engineered.csv", index=False)

In [67]:
# Checking that the logs are not droped
print("Raw rows:", len(pd.read_csv("../data_Lake/raw/Sample_Data_Ingestion.csv")))
print("Engineered rows:", len(df))

Raw rows: 200
Engineered rows: 200


In [68]:
# Unique Server Log Integrity Check
raw_df = pd.read_csv("../data_Lake/raw/Sample_Data_Ingestion.csv")

print("Raw unique servers:", raw_df["Server_ID"].nunique())
print("Engineered unique servers:", df["Server_ID"].nunique())

Raw unique servers: 200
Engineered unique servers: 200


In [70]:
# Timestamp Missing Flag Validation
df["timestamp_missing_flag"].value_counts()

timestamp_missing_flag
1    124
0     76
Name: count, dtype: int64